# 08 · Stability, Subgroup Audit and Findings

Bring model, capacity and source limitations together without presenting synthetic results as production evidence.

## Reading guide

This notebook is part of a connected workflow. It states the decision being made, shows the supporting checks and records limitations alongside the result. Source files are never modified in place.

In [1]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("FIFAR_DATA_DIR", PROJECT_ROOT / "data" / "raw" / "FiFAR"))
REPORTS = PROJECT_ROOT / "reports"
IMAGES = PROJECT_ROOT / "images"

sns.set_theme(style="whitegrid")
CORAL = "#F08FA0"
TEAL = "#0E6268"
DARK = "#15262B"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Set FIFAR_DATA_DIR to the extracted official FiFAR directory before running this notebook."
    )

In [2]:
base = pd.read_csv(DATA_ROOT / 'alert_data' / 'Base.csv').dropna().copy()
metrics = json.loads((REPORTS / 'model_metrics.json').read_text())
review = json.loads((REPORTS / 'review_strategy_metrics.json').read_text())

## 1. Temporal stability

In [3]:
monthly = base[~base.month.eq(4)].groupby("month")["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean")
monthly.assign(fraud_rate_percent=lambda frame: (frame["fraud_rate"] * 100).round(2))

,applications,fraud_cases,fraud_rate,fraud_rate_percent
month,,,,
0.0,132440,1500,0.011326,1.13
1.0,127620,1198,0.009387,0.94
2.0,136979,1198,0.008746,0.87
3.0,150936,1392,0.009222,0.92
5.0,119323,1411,0.011825,1.18
6.0,108168,1450,0.013405,1.34
7.0,96843,1428,0.014746,1.47


The observed fraud rate changes across the supplied months. This supports temporal monitoring and makes random-only validation inappropriate.

## 2. Audit fields

In [4]:
audit_fields = ["income", "customer_age", "employment_status", "housing_status"]
pd.DataFrame({
    "field": audit_fields,
    "used_by_primary_model": [False] * len(audit_fields),
    "retained_for_subgroup_audit": [True] * len(audit_fields),
})

,field,used_by_primary_model,retained_for_subgroup_audit
0,income,False,True
1,customer_age,False,True
2,employment_status,False,True
3,housing_status,False,True


Excluding these fields does not guarantee fairness. Remaining features may act as proxies, and the synthetic dataset cannot establish legal or regulatory suitability.

In [5]:
age_profile = base.groupby("customer_age")["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean")
age_profile.assign(fraud_rate_percent=lambda frame: (frame["fraud_rate"] * 100).round(2))

,applications,fraud_cases,fraud_rate,fraud_rate_percent
customer_age,,,,
10,18918,74,0.003912,0.39
20,223976,1205,0.005380,0.54
30,287301,2589,0.009011,0.90
40,219143,2876,0.013124,1.31
50,128844,2805,0.021771,2.18
60,31802,1149,0.036130,3.61
70,5941,263,0.044269,4.43
80,1178,64,0.054329,5.43
90,70,4,0.057143,5.71


In [6]:
employment_profile = base.groupby("employment_status")["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean")
employment_profile.assign(fraud_rate_percent=lambda frame: (frame["fraud_rate"] * 100).round(2))

,applications,fraud_cases,fraud_rate,fraud_rate_percent
employment_status,,,,
CA,669037,8899,0.013301,1.33
CB,129076,953,0.007383,0.74
CC,34123,932,0.027313,2.73
CD,23951,100,0.004175,0.42
CE,20621,53,0.002570,0.26
CF,39964,85,0.002127,0.21
CG,401,7,0.017456,1.75


These are descriptive target distributions, not model fairness metrics. Full subgroup evaluation requires saved test predictions, minimum-group support rules and uncertainty intervals.

## 3. Consolidated final-month result

In [7]:
selected = metrics["models"]["hist_gradient_boosting"]
three_percent = next(row for row in selected["test_capacity"] if row["review_share"] == .03)
pd.Series({
    "test_applications": metrics["split"]["test_rows"],
    "average_precision": selected["test"]["average_precision"],
    "roc_auc": selected["test"]["roc_auc"],
    **three_percent,
})

test_applications        96843.000000
average_precision            0.178916
roc_auc                      0.873893
review_share                 0.030000
review_capacity           2905.000000
fraud_captured             533.000000
precision_at_capacity        0.183477
recall_at_capacity           0.373249
dtype: float64

## 4. Alert-review result

In [8]:
pd.DataFrame(review['strategy_summary']).sort_values('mean_accuracy', ascending=False)

,strategy,scenarios,mean_accuracy,mean_precision,mean_recall,mean_false_positive,mean_false_negative,mean_human_reviews,mean_unused_capacity,accuracy_std,precision_std,recall_std,min_accuracy,max_accuracy
2,risk_band_specialist,25,0.609361,0.277829,0.900591,1670.40,70.68,4052.0,0.0,0.026458,0.014037,0.011688,0.574826,0.668162
0,global_skill,25,0.607871,0.277469,0.903404,1679.04,68.68,4052.0,0.0,0.027623,0.014786,0.009648,0.570563,0.662105
1,random_capacity,25,0.571541,0.261234,0.917300,1850.84,58.80,4052.0,0.0,0.028869,0.013216,0.014656,0.532197,0.632488


## 5. Findings

- The supplied application target is rare, so accuracy is not informative for the ranking model.
- Temporal evaluation exposes changing fraud prevalence.
- Gradient boosting improves ranking and fraud recovery over logistic regression.
- At three percent capacity, the queue captures 533 of 1,428 fraud cases.
- That queue still contains 2,372 legitimate applications and misses 895 fraud cases.
- Across 25 test teams, risk-band assignment reduces mean false positives by about 180 compared with random capacity allocation, while recall falls by about 1.7 percentage points.
- Analyst variation makes assignment and workload part of the detection problem.

## 6. Limitations

- All applications and analysts are synthetic.
- The supplied base CSV is truncated during month 4.
- The current model comparison is deliberately limited.
- No monetary loss or investigation cost is supplied.
- Protected-field exclusion does not prove fairness.
- Analyst skill is estimated from a short historical window and may change.
- Results do not demonstrate production or regulatory readiness.

## Recommendation

Use the model as a ranking component, not an automatic rejection rule. Monitor queue precision, fraud recovery, monthly drift, subgroup behaviour and analyst-assignment outcomes. Select the assignment objective only after defining the relative cost of false positives and missed fraud.